## Agents in LlamaIndex

In [1]:
%pip install llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface -U -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\sobha\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from huggingface_hub import login

login()

### Initialising agents
Let's start by initialising an agent. We will use the basic AgentWorkflow class to create an agent.

In [3]:
from huggingface_hub import AsyncInferenceClient
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult, AgentStream


class FixedHFInferenceAPI(HuggingFaceInferenceAPI):
    # llama-index closes the shared async client after each stream, so the agent's
    # second LLM call fails with "client has been closed". Use a fresh client per call.
    async def astream_chat(self, messages, **kwargs):
        self._async_client = AsyncInferenceClient(**self._get_inference_client_kwargs())
        return await super().astream_chat(messages, **kwargs)


def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b


llm = FixedHFInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[subtract, multiply, divide, add],
    llm=llm,
    system_prompt="You are a math agent that can add, subtract, multiply, and divide numbers using provided tools.",
)

Then, we can run the agent and get the response and reasoning behind the tool calls.

In [4]:
handler = agent.run("What is (2 + 2) * 2?")
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {"a": 2, "b": 2}
```
Called tool:  add {'a': 2, 'b': 2} => 4
Thought: Now I need to multiply the result by 2.
Action: multiply
Action Input: {'a': 4, 'b': 2}
Called tool:  multiply {'a': 4, 'b': 2} => 8
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: (2 + 2) * 2 = 8

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='(2 + 2) * 2 = 8')]), structured_response=None, current_agent_name='Agent', raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role=None, content='8', reasoning=None, tool_call_id=None, tool_calls=None), index=0, finish_reason=None, logprobs=None, token_ids=None)], created=1793119873, id='chatcmpl-2ad003e2-fda1-4168-8cfc-079a475a4e38', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=None, object='chat.completion.chunk'), tool_calls=[ToolCallResult(tool_name='add', tool_kwargs={'a': 2, 'b': 2}, tool_id='7571f477-7bdc-4426-9695-33d5b9f4309a', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='4')], tool_name='add', raw_input={'args': (), 'kwargs': {'a': 2, 'b': 2}}, raw_output=4, is_error=False), return_direct=False), ToolCallResult(tool_name='multiply'

In a similar fashion, we can pass state and context to the agent.

In [5]:
from llama_index.core.workflow import Context

ctx = Context(agent)

response = await agent.run("My name is Bob.", ctx=ctx)
response = await agent.run("What was my name again?", ctx=ctx)
response

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Your name is Bob.')]), structured_response=None, current_agent_name='Agent', raw=ChatCompletionStreamOutput(choices=[ChatCompletionStreamOutputChoice(delta=ChatCompletionStreamOutputDelta(role=None, content='.', reasoning=None, tool_call_id=None, tool_calls=None), index=0, finish_reason=None, logprobs=None, token_ids=None)], created=1793119899, id='chatcmpl-e57f6d56-5304-4f9d-93b7-dc902da082c7', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=None, object='chat.completion.chunk'), tool_calls=[], retry_messages=[])